# Session 10 · Non-Linear Regression Analysis

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

## Data: a curved relationship

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
rng = np.random.default_rng(10)
x = rng.uniform(-3, 3, 120)
y = 0.8 * x ** 2 - x + 2 + rng.normal(0, 0.8, 120)
X = x.reshape(-1, 1)

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

## Example 1 - a line under-fits

In [2]:
lin = LinearRegression().fit(X, y)
print("linear R2:", round(r2_score(y, lin.predict(X)), 3))

linear R2: 0.336


## Example 2 - polynomial regression

In [3]:
poly2 = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression()).fit(X, y)
print("quadratic R2:", round(r2_score(y, poly2.predict(X)), 3))
print("coefficients:", poly2[-1].coef_.round(2), "intercept:", round(poly2[-1].intercept_, 2))
assert r2_score(y, poly2.predict(X)) > r2_score(y, lin.predict(X))

quadratic R2: 0.903
coefficients: [-1.01  0.84] intercept: 1.7


## Example 3 - choose the degree with held-out data

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
for d in [1, 2, 3, 5, 9, 15]:
    m = make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()).fit(X_tr, y_tr)
    print(f"degree {d:>2}: train RMSE {rmse(y_tr, m.predict(X_tr)):.3f} | test RMSE {rmse(y_te, m.predict(X_te)):.3f}")

degree  1: train RMSE 2.265 | test RMSE 1.836
degree  2: train RMSE 0.820 | test RMSE 0.828
degree  3: train RMSE 0.815 | test RMSE 0.839
degree  5: train RMSE 0.795 | test RMSE 0.847
degree  9: train RMSE 0.794 | test RMSE 0.848
degree 15: train RMSE 0.747 | test RMSE 0.893


## Example 4 - exponential decay (phone resale value)

In [5]:
from scipy.optimize import curve_fit
months = np.arange(0, 37, 3)
price = 150_000 * np.exp(-0.045 * months) * rng.normal(1, 0.03, months.size)

def decay(t, a, b):
    return a * np.exp(-b * t)

(a, b), _ = curve_fit(decay, months, price, p0=[150_000, 0.05])
print(f"a = {a:,.0f}, b = {b:.4f}, value halves every {np.log(2) / b:.1f} months")
assert abs(b - 0.045) < 0.01

a = 146,858, b = 0.0446, value halves every 15.5 months


## Example 5 - logistic growth (app users)

In [6]:
weeks = np.arange(0, 30)
users = 50_000 / (1 + np.exp(-0.4 * (weeks - 12))) + rng.normal(0, 800, weeks.size)

def logistic(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))

(L, k, t0), _ = curve_fit(logistic, weeks, users, p0=[users.max(), 0.3, weeks.mean()])
print(f"saturation ~{L:,.0f} users, fastest growth at week {t0:.1f}")

saturation ~49,447 users, fastest growth at week 12.0


## Example 6 - straighten with logs (power law)

In [7]:
area = rng.uniform(500, 3000, 80)
house_price = 9000 * area ** 1.1 * rng.normal(1, 0.08, 80)
lm = LinearRegression().fit(np.log(area).reshape(-1, 1), np.log(house_price))
print("estimated exponent b:", round(lm.coef_[0], 3))

estimated exponent b: 1.122
